# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Color Image Processing and Vector-Valued Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/05_ColorImageProcessing.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
Unlike scalar monochromatic images where each pixel is an intensity value $f(x,y) \in \mathbb{R}$, color images are **vector fields** $\mathbf{f}(x,y) = [f_1(x,y), f_2(x,y), f_3(x,y)]^T \in \mathbb{R}^3$. Processing multi-channel data requires specialized vector-valued operators because marginal (per-channel) scalar processing often leads to severe chromatic distortions, false edge generation, and out-of-gamut artifacts.

By completing this notebook, you will master:
1. **Vector-Valued Representations**: Multi-spectral tensor layouts (`HWC` vs. `CHW`), dynamic range normalization, and channel decomposition.
2. **Color Spaces & Conversions**: Additive (RGB) vs. Subtractive (CMYK), geometric HSI cylindrical color modeling, and exact round-trip validation.
3. **Pseudocoloring & Non-Linear Tone Mapping**: Intensity slicing, sinusoidal color LUTs, and why per-channel Gamma induces severe chromatic distortion compared to intensity-domain Gamma.
4. **Vector-Valued Spatial Operations**: Marginal vs. intensity-domain filtering, out-of-gamut clipping metrology, and the **Di Zenzo Structure Tensor** for detecting iso-luminant chromatic boundaries.
5. **Colorimetric Foundations**: Radiometric linearization, sRGB EOTF, CIE 1931 $(x,y)$ chromaticity space, CIELAB, $\Delta E_{76}$, and CIEDE2000 ($\Delta E_{00}$).
6. **Gamut Mapping & Chromatic Adaptation**: Soft vs. hard gamut clipping, Gray-World, White-Patch Retinex, and von Kries color constancy.
7. **Vector Filtering & Reconstruction**: Vector Median Filtering (VMF), Vector Bilateral, Non-Local Means (NLM), Bayer CFA mosaic simulation, and Malvar-He-Cutler demosaicing.
8. **End-to-End Capstone ISP Pipeline**: A complete Camera Image Signal Processor integrating raw CFA capture through demosaicing, white balancing, denoising, sharpening, gamut mapping, and sRGB output encoding.

---


---
## SECTION 0: Environment Setup & Library Asset Loader

### Theoretical & Computational Foundations
Vector-valued operations on 3-channel color tensors require floating-point representations (`np.float64`) normalized to $[0.0, 1.0]$. In this section, we import standard libraries (`numpy`, `scipy`, `matplotlib`, `skimage`, `cv2`), configure publication-grade plot parameters, and define a dictionary providing direct access to standard benchmark datasets from `skimage.data`.


In [ ]:
# Section 0: Environment Setup & Library Asset Loader
import os
import sys
import numpy as np
import scipy.ndimage as ndimage
import scipy.signal as signal
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from skimage import data, color, transform, filters, exposure, util, restoration, metrics
import cv2
from PIL import Image

# Configure publication-grade styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9.5
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10.5
plt.rcParams['xtick.labelsize'] = 8.5
plt.rcParams['ytick.labelsize'] = 8.5

# Test dataset loader dictionary using skimage.data
def load_benchmark_datasets():
    datasets = {
        'astronaut': util.img_as_float64(data.astronaut()),
        'chelsea': util.img_as_float64(data.chelsea()),
        'retina': util.img_as_float64(data.retina()),
        'colorwheel': util.img_as_float64(data.colorwheel()),
        'coffee': util.img_as_float64(data.coffee()),
        'camera': util.img_as_float64(data.camera()),
        'moon': util.img_as_float64(data.moon()),
        'cat': util.img_as_float64(data.cat())
    }
    return datasets

img_assets = load_benchmark_datasets()
print(f"Environment successfully initialized. Available datasets: {list(img_assets.keys())}")


---
## SECTION 1: Vector-Valued Image Representation & Channel Decomposition

### 1.1 Vector-Valued Image Definition
A 2D scalar image is a scalar field $f: \Omega \subset \mathbb{Z}^2 \to \mathbb{R}$. In contrast, a 2D trichromatic color image is a **vector field**:
$$\mathbf{f}(x,y) = \begin{bmatrix} R(x,y) \\ G(x,y) \\ B(x,y) \end{bmatrix} \in \mathbb{R}^3$$
where each spatial coordinate $(x,y)$ maps to a 3-dimensional vector in a color coordinate space.

### 1.2 Memory Layouts and Dynamic Range
- **Memory Tensors**: In NumPy/OpenCV, color images are stored in **HWC (Height $\times$ Width $\times$ Channels)** layout ($M \times N \times 3$). In PyTorch/Deep Learning frameworks, **CHW (Channels $\times$ Height $\times$ Width)** layout ($3 \times M \times N$) is standard for GPU memory coalescing.
- **Dynamic Range Hygiene**: Discrete 8-bit integer formats (`uint8` $\in [0, 255]$) are susceptible to underflow and overflow under linear algebraic operations. Normalization to standard IEEE 754 64-bit floating point (`float64` $\in [0.0, 1.0]$) ensures numerical stability.


In [ ]:
# Cell 1.1: Tensor Inspection and Normalization
img_astro = img_assets['astronaut']

print("=== Vector-Valued Image Properties ===")
print(f"Spatial Dimensions (H x W): {img_astro.shape[0]} x {img_astro.shape[1]}")
print(f"Number of Spectral Channels: {img_astro.shape[2]}")
print(f"Tensor Rank: {img_astro.ndim}D")
print(f"Data Type: {img_astro.dtype}")
print(f"Dynamic Range: [{img_astro.min():.4f}, {img_astro.max():.4f}]")
print(f"Memory Layout: HWC (Bytes: {img_astro.nbytes} bytes)")


In [ ]:
# Cell 1.2: Channel Decomposition and Color Embedding Visualization
R_chan = img_astro[:, :, 0]
G_chan = img_astro[:, :, 1]
B_chan = img_astro[:, :, 2]

# Isolated pure color channel vectors
R_color = np.zeros_like(img_astro)
R_color[:, :, 0] = R_chan

G_color = np.zeros_like(img_astro)
G_color[:, :, 1] = G_chan

B_color = np.zeros_like(img_astro)
B_color[:, :, 2] = B_chan

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: Grayscale scalar projections
axes[0, 0].imshow(R_chan, cmap='gray')
axes[0, 0].set_title(r"(A) Red Channel Intensity $R(x,y)$", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(G_chan, cmap='gray')
axes[0, 1].set_title(r"(B) Green Channel Intensity $G(x,y)$", fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(B_chan, cmap='gray')
axes[0, 2].set_title(r"(C) Blue Channel Intensity $B(x,y)$", fontweight='bold')
axes[0, 2].axis('off')

# Row 2: Isolated color vector embeddings
axes[1, 0].imshow(R_color)
axes[1, 0].set_title(r"(D) Isolated Red Vector $[R, 0, 0]^T$", fontweight='bold', color='darkred')
axes[1, 0].axis('off')

axes[1, 1].imshow(G_color)
axes[1, 1].set_title(r"(E) Isolated Green Vector $[0, G, 0]^T$", fontweight='bold', color='darkgreen')
axes[1, 1].axis('off')

axes[1, 2].imshow(B_color)
axes[1, 2].set_title(r"(F) Isolated Blue Vector $[0, 0, B]^T$", fontweight='bold', color='darkblue')
axes[1, 2].axis('off')

plt.suptitle("Figure 1.1: Multi-Channel Vector Decomposition: Scalar Grayscale Projections vs. Pure Color Embeddings",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 2: Additive vs. Subtractive Mixing & CMYK Separations

### 2.1 Additive Primary Mixing (RGB)
In emissive displays (CRTs, LCDs, OLEDs), colors are generated by **additive spectral superposition**:
$$\mathbf{C}_{\text{additive}} = R \cdot \mathbf{P}_R + G \cdot \mathbf{P}_G + B \cdot \mathbf{P}_B$$
- Red + Green = **Yellow** ($[1, 1, 0]^T$)
- Red + Blue = **Magenta** ($[1, 0, 1]^T$)
- Green + Blue = **Cyan** ($[0, 1, 1]^T$)
- Red + Green + Blue = **White** ($[1, 1, 1]^T$)

### 2.2 Subtractive Primary Mixing (CMY & CMYK)
In reflective media (pigments, printing inks), colorants act as optical band-stop absorption filters:
$$\begin{bmatrix} C \\ M \\ Y \end{bmatrix} = \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix} - \begin{bmatrix} R \\ G \\ B \end{bmatrix}$$

Because printing equal amounts of pure $C, M, Y$ inks yields a muddy dark brown rather than a deep neutral black (due to chemical impurities), commercial printing utilizes **Under-Color Removal (UCR)** to extract a dedicated Key/Black channel ($K$):
$$K = \min(C, M, Y)$$
$$C' = \frac{C - K}{1 - K + \epsilon}, \quad M' = \frac{M - K}{1 - K + \epsilon}, \quad Y' = \frac{Y - K}{1 - K + \epsilon}$$
where $\epsilon = 10^{-7}$ guards against division-by-zero on pure black pixels ($K=1$).


In [ ]:
# Cell 2.1: Additive vs. Subtractive Primary Mixing Simulation
dim = 300
y, x = np.ogrid[:dim, :dim]

# Disk centers
c_top = (100, 150)
c_left = (190, 105)
c_right = (190, 195)
radius = 75

mask_top = ((y - c_top[0])**2 + (x - c_top[1])**2) <= radius**2
mask_left = ((y - c_left[0])**2 + (x - c_left[1])**2) <= radius**2
mask_right = ((y - c_right[0])**2 + (x - c_right[1])**2) <= radius**2

# Additive RGB Canvas (Black Background)
add_canvas = np.zeros((dim, dim, 3), dtype=np.float64)
add_canvas[mask_top, 0] += 1.0   # Red
add_canvas[mask_left, 1] += 1.0  # Green
add_canvas[mask_right, 2] += 1.0 # Blue
add_canvas = np.clip(add_canvas, 0.0, 1.0)

# Subtractive CMY Canvas (White Background)
sub_canvas = np.ones((dim, dim, 3), dtype=np.float64)
# Cyan absorbs Red (removes R)
sub_canvas[mask_top, 0] -= 1.0
# Magenta absorbs Green (removes G)
sub_canvas[mask_left, 1] -= 1.0
# Yellow absorbs Blue (removes B)
sub_canvas[mask_right, 2] -= 1.0
sub_canvas = np.clip(sub_canvas, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].imshow(add_canvas)
axes[0].set_title("(A) Additive Color Mixing (RGB Light Emissive)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(sub_canvas)
axes[1].set_title("(B) Subtractive Color Mixing (CMY Pigment Absorption)", fontweight='bold')
axes[1].axis('off')

plt.suptitle("Figure 2.1: Physical Duality of Additive Superposition vs. Subtractive Spectral Absorption",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 2.2: Vectorized RGB to CMYK Conversion and Plate Separation
def rgb_to_cmyk(img):
    # Vectorized conversion from normalized RGB [0,1] to CMYK [0,1]
    C = 1.0 - img[:, :, 0]
    M = 1.0 - img[:, :, 1]
    Y = 1.0 - img[:, :, 2]
    
    K = np.min(np.stack([C, M, Y], axis=-1), axis=-1)
    
    eps = 1e-7
    denom = 1.0 - K + eps
    C_prime = (C - K) / denom
    M_prime = (M - K) / denom
    Y_prime = (Y - K) / denom
    
    return np.stack([C_prime, M_prime, Y_prime, K], axis=-1)

img_chelsea = img_assets['chelsea']
cmyk_plates = rgb_to_cmyk(img_chelsea)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
plate_names = ['Cyan (C\')', 'Magenta (M\')', 'Yellow (Y\')', 'Key / Black (K)']
cmaps = ['Blues_r', 'Purples_r', 'YlOrBr_r', 'gray_r']

for idx in range(4):
    im = axes[idx].imshow(cmyk_plates[:, :, idx], cmap='gray')
    axes[idx].set_title(f"Plate: {plate_names[idx]}", fontweight='bold')
    axes[idx].axis('off')
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

plt.suptitle("Figure 2.2: CMYK Color Separation Plates for Chelsea via Under-Color Removal (UCR)",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 3: Custom Vectorized RGB $\leftrightarrow$ HSI Conversions & Round-Trip Validation

### 3.1 Geometric Representation of HSI Space
While RGB is hardware-oriented, **HSI (Hue, Saturation, Intensity)** separates chromaticity from luminance to mimic human visual perception:
1. **Intensity ($I \in [0, 1]$)**: The average monochromatic brightness:
   $$I = \frac{1}{3}(R + G + B)$$
2. **Saturation ($S \in [0, 1]$)**: The purity or vividness of the color relative to white/gray:
   $$S = 1 - \frac{3}{R + G + B + \epsilon} \min(R, G, B)$$
3. **Hue ($H \in [0^\circ, 360^\circ)$)**: The dominant spectral wavelength expressed as an angular coordinate:
   $$\theta = \arccos\left( \frac{\frac{1}{2}[(R-G) + (R-B)]}{\sqrt{(R-G)^2 + (R-B)(G-B)} + \epsilon} \right)$$
   $$H = \begin{cases} \theta & \text{if } B \le G \\ 360^\circ - \theta & \text{if } B > G \end{cases}$$

### 3.2 Inverse HSI to RGB Reconstruction (Sector Trigonometry)
Given $(H, S, I)$, RGB coordinates are reconstructed based on the $120^\circ$ Hue sector:
- **RG Sector ($0^\circ \le H < 120^\circ$)**:
  $$B = I(1 - S), \quad R = I\left[ 1 + \frac{S \cos H}{\cos(60^\circ - H)} \right], \quad G = 3I - (R + B)$$
- **GB Sector ($120^\circ \le H < 240^\circ$)** with $H' = H - 120^\circ$:
  $$R = I(1 - S), \quad G = I\left[ 1 + \frac{S \cos H'}{\cos(60^\circ - H')} \right], \quad B = 3I - (R + G)$$
- **BR Sector ($240^\circ \le H \le 360^\circ$)** with $H' = H - 240^\circ$:
  $$G = I(1 - S), \quad B = I\left[ 1 + \frac{S \cos H'}{\cos(60^\circ - H')} \right], \quad R = 3I - (G + B)$$


In [ ]:
# Cell 3.1 & 3.2: Vectorized RGB <-> HSI Engine
def rgb_to_hsi(rgb_img, eps=1e-12):
    # Vectorized RGB to HSI conversion with full numerical stability
    R = rgb_img[:, :, 0]
    G = rgb_img[:, :, 1]
    B = rgb_img[:, :, 2]
    I = (R + G + B) / 3.0
    min_val = np.minimum(np.minimum(R, G), B)
    
    # Saturation
    sum_rgb = R + G + B
    S = np.zeros_like(I)
    nonzero_mask = sum_rgb > 1e-9
    S[nonzero_mask] = 1.0 - (3.0 / sum_rgb[nonzero_mask]) * min_val[nonzero_mask]
    S = np.clip(S, 0.0, 1.0)
    
    # Hue
    num = 0.5 * ((R - G) + (R - B))
    den = np.sqrt((R - G)**2 + (R - B) * (G - B))
    val = np.zeros_like(I)
    valid_chroma = (den > 1e-9) & (S > 1e-6)
    val[valid_chroma] = np.clip(num[valid_chroma] / den[valid_chroma], -1.0, 1.0)
    theta = np.arccos(val)
    
    H = np.zeros_like(theta)
    mask_b_le_g = (B <= G)
    H[mask_b_le_g] = theta[mask_b_le_g]
    H[~mask_b_le_g] = 2.0 * np.pi - theta[~mask_b_le_g]
    
    H_deg = np.degrees(H) % 360.0
    H_deg[~valid_chroma] = 0.0
    return np.stack([H_deg, S, I], axis=-1)

def hsi_to_rgb(hsi_img):
    # Vectorized HSI to RGB conversion via sector-wise trigonometry
    H_deg = hsi_img[:, :, 0] % 360.0
    S = np.clip(hsi_img[:, :, 1], 0.0, 1.0)
    I = np.clip(hsi_img[:, :, 2], 0.0, 1.0)
    
    H_rad = np.radians(H_deg)
    R = np.zeros_like(I)
    G = np.zeros_like(I)
    B = np.zeros_like(I)
    
    # Achromatic (S == 0)
    achromatic = (S < 1e-6)
    R[achromatic] = I[achromatic]
    G[achromatic] = I[achromatic]
    B[achromatic] = I[achromatic]
    
    chromatic = ~achromatic
    
    # Sector 1: RG Sector (0 <= H < 120)
    s1 = chromatic & (H_deg >= 0.0) & (H_deg < 120.0)
    B[s1] = I[s1] * (1.0 - S[s1])
    R[s1] = I[s1] * (1.0 + (S[s1] * np.cos(H_rad[s1])) / np.cos(np.radians(60.0) - H_rad[s1]))
    G[s1] = 3.0 * I[s1] - (R[s1] + B[s1])
    
    # Sector 2: GB Sector (120 <= H < 240)
    s2 = chromatic & (H_deg >= 120.0) & (H_deg < 240.0)
    H2 = H_rad[s2] - np.radians(120.0)
    R[s2] = I[s2] * (1.0 - S[s2])
    G[s2] = I[s2] * (1.0 + (S[s2] * np.cos(H2)) / np.cos(np.radians(60.0) - H2))
    B[s2] = 3.0 * I[s2] - (R[s2] + G[s2])
    
    # Sector 3: BR Sector (240 <= H < 360)
    s3 = chromatic & (H_deg >= 240.0) & (H_deg <= 360.0)
    H3 = H_rad[s3] - np.radians(240.0)
    G[s3] = I[s3] * (1.0 - S[s3])
    B[s3] = I[s3] * (1.0 + (S[s3] * np.cos(H3)) / np.cos(np.radians(60.0) - H3))
    R[s3] = 3.0 * I[s3] - (G[s3] + B[s3])
    
    rgb = np.stack([R, G, B], axis=-1)
    return np.clip(rgb, 0.0, 1.0)


In [ ]:
# Cell 3.3 & 3.4: Round-Trip Numerical Validation and HSI Channel Display
img_retina = img_assets['retina']
hsi_retina = rgb_to_hsi(img_retina)
recon_retina = hsi_to_rgb(hsi_retina)

# Numerical precision validation
max_abs_err = np.max(np.abs(img_retina - recon_retina))
print(f"=== HSI Round-Trip Invertibility Validation ===")
print(f"Maximum Reconstruction Error: {max_abs_err:.2e}")
assert max_abs_err < 1e-6, "HSI inversion failed numerical tolerance!"
print("Assertion Passed: RGB -> HSI -> RGB is mathematically exact and reversible.")

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_retina)
axes[0].set_title("(A) Input RGB Image (Retina)", fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(hsi_retina[:, :, 0], cmap='twilight', vmin=0, vmax=360)
axes[1].set_title(r"(B) Hue Channel $H \in [0^\circ, 360^\circ)$", fontweight='bold')
axes[1].axis('off')
cbar1 = plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_ticks([0, 90, 180, 270, 360])
cbar1.set_ticklabels(['0°', '90°', '180°', '270°', '360°'])

im2 = axes[2].imshow(hsi_retina[:, :, 1], cmap='viridis', vmin=0, vmax=1)
axes[2].set_title(r"(C) Saturation Channel $S \in [0, 1]$", fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(hsi_retina[:, :, 2], cmap='gray', vmin=0, vmax=1)
axes[3].set_title(r"(D) Intensity Channel $I \in [0, 1]$", fontweight='bold')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.suptitle("Figure 3.1: HSI Color Decomposition of Retinal Vasculature showing Decoupled Chromaticity and Luminance",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 4: Pseudocolor Processing: Slicing & Sinusoidal Transformations

### 4.1 Theory of Scalar-to-Vector Mapping
The human visual system can distinguish only $\approx 30$ to $50$ shades of gray under constant adaptation, but can discern thousands of distinct color hues. **Pseudocolor processing** assigns colors to scalar monochromatic intensities $r(x,y) \in [0, 1]$:
1. **Intensity-Level Slicing**: Partitions the continuous intensity range into $P$ discrete sub-intervals, mapping each band to a constant color vector:
   $$\mathbf{c}(r) = \mathbf{C}_k \quad \text{for } r \in [l_{k-1}, l_k)$$
2. **Phase-Shifted Sinusoidal Transformation Functions**: Generates continuous, smooth pseudo-color palettes via trigonometric phase modulation:
   $$f_R(r) = \frac{1}{2}\left[ 1 + \sin(\pi r + \phi_R) \right]$$
   $$f_G(r) = \frac{1}{2}\left[ 1 + \sin(\pi r + \phi_G) \right]$$
   $$f_B(r) = \frac{1}{2}\left[ 1 + \sin(\pi r + \phi_B) \right]$$
   Setting phase offsets $(\phi_R, \phi_G, \phi_B) = (0, \pi/2, \pi)$ produces smooth spectral transitions spanning Blue $\to$ Cyan $\to$ Yellow $\to$ Red.


In [ ]:
# Cell 4.1: Intensity Slicing on Lunar Surface Image
img_moon = img_assets['moon']

# 8-band Staircase LUT
num_bands = 8
bands = np.linspace(0, 1, num_bands + 1)
palette = plt.get_cmap('jet', num_bands)(np.linspace(0, 1, num_bands))[:, :3]

pseudo_sliced = np.zeros((*img_moon.shape, 3), dtype=np.float64)
for i in range(num_bands):
    mask = (img_moon >= bands[i]) & (img_moon < bands[i+1])
    pseudo_sliced[mask] = palette[i]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_moon, cmap='gray')
axes[0].set_title("(A) Monochromatic Lunar Image $r(x,y)$", fontweight='bold')
axes[0].axis('off')

# Plot LUT curves
r_vals = np.linspace(0, 1, 500)
lut_vals = np.zeros((len(r_vals), 3))
for i in range(num_bands):
    mask = (r_vals >= bands[i]) & (r_vals <= bands[i+1])
    lut_vals[mask] = palette[i]

axes[1].plot(r_vals, lut_vals[:, 0], 'r-', label='Red LUT', linewidth=2)
axes[1].plot(r_vals, lut_vals[:, 1], 'g-', label='Green LUT', linewidth=2)
axes[1].plot(r_vals, lut_vals[:, 2], 'b-', label='Blue LUT', linewidth=2)
axes[1].set_title("(B) 8-Band Piecewise Constant Staircase LUT", fontweight='bold')
axes[1].set_xlabel("Input Intensity $r \in [0, 1]$")
axes[1].set_ylabel("Assigned Color Channel Output")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='center right')

axes[2].imshow(pseudo_sliced)
axes[2].set_title("(C) Intensity-Sliced Pseudocolor Map", fontweight='bold')
axes[2].axis('off')

plt.suptitle("Figure 4.1: Discrete Intensity-Level Slicing for Morphological Lunar Topography Segmentation",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 4.2: Continuous Sinusoidal Pseudocoloring
img_cam = img_assets['camera']

# Phase-shifted sinusoidal transformation profiles
phi_R = 0.0
phi_G = np.pi / 2.0
phi_B = np.pi

R_sin = 0.5 * (1.0 + np.sin(np.pi * img_cam + phi_R))
G_sin = 0.5 * (1.0 + np.sin(np.pi * img_cam + phi_G))
B_sin = 0.5 * (1.0 + np.sin(np.pi * img_cam + phi_B))

pseudo_sin = np.stack([R_sin, G_sin, B_sin], axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_cam, cmap='gray')
axes[0].set_title("(A) Monochromatic Input (Cameraman)", fontweight='bold')
axes[0].axis('off')

r_axis = np.linspace(0, 1, 300)
axes[1].plot(r_axis, 0.5 * (1.0 + np.sin(np.pi * r_axis + phi_R)), 'r-', label=r'$f_R(r): \phi_R=0$', linewidth=2)
axes[1].plot(r_axis, 0.5 * (1.0 + np.sin(np.pi * r_axis + phi_G)), 'g-', label=r'$f_G(r): \phi_G=\pi/2$', linewidth=2)
axes[1].plot(r_axis, 0.5 * (1.0 + np.sin(np.pi * r_axis + phi_B)), 'b-', label=r'$f_B(r): \phi_B=\pi$', linewidth=2)
axes[1].set_title("(B) Sinusoidal Transformation Curves", fontweight='bold')
axes[1].set_xlabel("Input Intensity $r$")
axes[1].set_ylabel("Output Intensity")
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='center right')

axes[2].imshow(pseudo_sin)
axes[2].set_title("(C) Smooth Sinusoidal Pseudocolored Output", fontweight='bold')
axes[2].axis('off')

plt.suptitle("Figure 4.2: Smooth Continuous Phase-Shifted Sinusoidal Pseudocoloring for Enhanced Dynamic Range Perception",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 5: Non-Linear Tone Mapping: Per-Channel vs. Intensity-Domain Gamma

### 5.1 Naive Per-Channel vs. Intensity-Domain Tone Mapping
Applying a non-linear power-law (Gamma) transformation $\mathcal{T}(r) = r^\gamma$ to color images can be performed via two fundamentally different strategies:
1. **Marginal (Per-Channel) RGB Gamma**:
   $$\mathbf{g}_{\text{marginal}}(x,y) = \begin{bmatrix} R(x,y)^\gamma \\ G(x,y)^\gamma \\ B(x,y)^\gamma \end{bmatrix}$$
   *Failure Mechanism*: Unless $R=G=B$, exponentiation alters the inter-channel ratios $\frac{R}{G}, \frac{G}{B}$, causing severe **angular chromatic shift (Hue drift)** and unnatural color distortion.
2. **Intensity-Domain Gamma**:
   Transform to HSI space, apply Gamma exclusively to the Intensity channel ($I \to I^\gamma$), and preserve Hue ($H$) and Saturation ($S$) unchanged:
   $$\mathbf{g}_{\text{intensity}}(x,y) = \text{hsi\_to\_rgb}\left( \begin{bmatrix} H(x,y) \\ S(x,y) \\ I(x,y)^\gamma \end{bmatrix} \right)$$

### 5.2 Angular Chromatic Distortion (Hue Shift Metric)
The spatial chromatic error is quantified by the angular distance on the circular Hue manifold:
$$\Delta H(x,y) = \min\big(|H_{\text{orig}}(x,y) - H_{\text{proc}}(x,y)|, \, 360^\circ - |H_{\text{orig}}(x,y) - H_{\text{proc}}(x,y)|\big)$$


In [ ]:
# Cell 5.1 & 5.2: Tone Mapping and Hue Shift Metrology
gamma_val = 0.45 # Brightness expansion for underexposed scene
img_astro = img_assets['astronaut']

# 1. Marginal RGB Gamma
rgb_gamma = np.power(img_astro, gamma_val)

# 2. Intensity-Domain Gamma in HSI
hsi_orig = rgb_to_hsi(img_astro)
hsi_gamma = np.copy(hsi_orig)
hsi_gamma[:, :, 2] = np.power(hsi_gamma[:, :, 2], gamma_val)
intensity_gamma = hsi_to_rgb(hsi_gamma)

# Compute Hue Shift Delta H
hsi_marginal = rgb_to_hsi(rgb_gamma)
h_orig = hsi_orig[:, :, 0]
h_marg = hsi_marginal[:, :, 0]

raw_diff = np.abs(h_orig - h_marg)
delta_h_marginal = np.minimum(raw_diff, 360.0 - raw_diff)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].imshow(rgb_gamma)
axes[0, 0].set_title(f"(A) Marginal RGB Gamma ($\gamma={gamma_val}$)\n[Visible Color Shifts & Skin Tone Drift]", fontweight='bold', color='crimson')
axes[0, 0].axis('off')

axes[0, 1].imshow(intensity_gamma)
axes[0, 1].set_title(f"(B) Intensity-Domain HSI Gamma ($\gamma={gamma_val}$)\n[True Chromatic Fidelity Preserved]", fontweight='bold', color='darkgreen')
axes[0, 1].axis('off')

im2 = axes[1, 0].imshow(delta_h_marginal, cmap='hot', vmin=0, vmax=60)
axes[1, 0].set_title(r"(C) Spatial Hue Error Map $\Delta H(x,y)$ (Degrees)", fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

# Hue Error Histogram
axes[1, 1].hist(delta_h_marginal.ravel(), bins=60, range=(0.1, 60), color='crimson', edgecolor='black', alpha=0.7)
axes[1, 1].set_title(r"(D) Hue Drift Distribution ($\bar{\Delta H} = " + f"{np.mean(delta_h_marginal):.2f}^\circ$)", fontweight='bold')
axes[1, 1].set_xlabel("Angular Hue Shift (Degrees)")
axes[1, 1].set_ylabel("Pixel Count")
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("Figure 5.1: Comparative Tone Mapping: Severe Chromatic Distortion in Per-Channel Gamma vs. True Fidelity in HSI",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 6: Color Complements & Chromatic Histogram Transformations

### 6.1 Mathematical Complements
The color complement of a chromatic vector can be computed via two distinct formulations:
1. **Euclidean RGB Inversion**:
   $$\mathbf{f}_{\text{inv}}(x,y) = \mathbf{1} - \mathbf{f}(x,y) = \begin{bmatrix} 1 - R \\ 1 - G \\ 1 - B \end{bmatrix}$$
2. **Polar HSI Chromatic Complement**:
   Rotating Hue by $180^\circ$ on the color circle while preserving Saturation ($S$) and Intensity ($I$):
   $$H_{\text{comp}}(x,y) = \big(H(x,y) + 180^\circ\big) \pmod{360^\circ}$$
   $$\mathbf{f}_{\text{hsi\_comp}}(x,y) = \text{hsi\_to\_rgb}\left( \begin{bmatrix} H_{\text{comp}} \\ S \\ I \end{bmatrix} \right)$$


In [ ]:
# Cell 6.1 & 6.2: Color Complement Mechanisms on Colorwheel
img_wheel = img_assets['colorwheel']

# 1. RGB Inversion
comp_rgb = 1.0 - img_wheel

# 2. HSI 180° Hue Rotation
hsi_wheel = rgb_to_hsi(img_wheel)
hsi_comp = np.copy(hsi_wheel)
hsi_comp[:, :, 0] = (hsi_comp[:, :, 0] + 180.0) % 360.0
comp_hsi = hsi_to_rgb(hsi_comp)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_wheel)
axes[0].set_title("(A) Original Colorwheel", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(comp_rgb)
axes[1].set_title("(B) RGB Euclidean Complement ($\mathbf{1} - \mathbf{f}$)\n[Inverts both Color & Intensity]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(comp_hsi)
axes[2].set_title("(C) HSI Polar Complement ($H + 180^\circ$)\n[Rotates Hue, Keeps Intensity Constant]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 6.1: Color Complement Duality: Total RGB Inversion vs. Pure Chromatic HSI Rotation",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 7: Spatial Filtering: Marginal vs. Intensity-Domain Operations

### 7.1 Spatial Smoothing and Color Shift
- **Marginal Smoothing**: Convolving each channel independently with kernel $h$: $\mathbf{f}_{\text{smooth}} = [R*h, G*h, B*h]^T$. Across sharp chromatic boundaries, mixing red and green channels produces an artificial brownish fringe.
- **Intensity-Domain Smoothing**: Filtering only $I$ ($[H, S, I*h]^T$) eliminates spatial resolution without introducing unnatural intermediate hues.

### 7.2 Laplacian Sharpening and Out-of-Gamut Clipping
The second-order isotropic Laplacian sharpening operator is:
$$\mathbf{f}_{\text{sharp}} = \mathbf{f} - c \nabla^2 \mathbf{f}$$
When applied marginally to RGB channels, overshoot and undershoot along edges frequently violate the physically valid color gamut $[0.0, 1.0]$. The **Out-of-Gamut Clipping Percentage** is:
$$\%_{\text{clipped}} = \frac{1}{3MN} \sum_{c \in \{R,G,B\}} \sum_{x,y} \mathbb{I}\big(c(x,y) < 0 \text{ or } c(x,y) > 1\big) \times 100\%$$


In [ ]:
# Cell 7.1 & 7.2: Marginal vs. Intensity Spatial Filtering and Gamut Clipping
img_astro = img_assets['astronaut']

# Laplacian Sharpening Kernel
c_sharp = 0.6
laplace_k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)

# 1. Marginal RGB Sharpening
sharp_rgb_raw = np.zeros_like(img_astro)
for c in range(3):
    lap = ndimage.convolve(img_astro[:, :, c], laplace_k)
    sharp_rgb_raw[:, :, c] = img_astro[:, :, c] - c_sharp * lap

clipped_mask_rgb = (sharp_rgb_raw < 0.0) | (sharp_rgb_raw > 1.0)
clip_pct_rgb = (np.sum(clipped_mask_rgb) / img_astro.size) * 100.0
sharp_rgb = np.clip(sharp_rgb_raw, 0.0, 1.0)

# 2. Intensity-Domain Sharpening in HSI
hsi_astro = rgb_to_hsi(img_astro)
I_lap = ndimage.convolve(hsi_astro[:, :, 2], laplace_k)
I_sharp = hsi_astro[:, :, 2] - c_sharp * I_lap

hsi_sharp = np.copy(hsi_astro)
hsi_sharp[:, :, 2] = np.clip(I_sharp, 0.0, 1.0)
sharp_hsi = hsi_to_rgb(hsi_sharp)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Zoom on astronaut collar / helmet edge
r1, r2, c1, c2 = 120, 220, 150, 250
axes[0].imshow(img_astro[r1:r2, c1:c2])
axes[0].set_title("(A) Original Detail Crop", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(sharp_rgb[r1:r2, c1:c2])
axes[1].set_title(f"(B) Marginal RGB Sharpening\n[Out-of-Gamut Clipped: {clip_pct_rgb:.2f}%]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(sharp_hsi[r1:r2, c1:c2])
axes[2].set_title("(C) Intensity-Domain HSI Sharpening\n[Zero Chromatic Aberration]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

plt.suptitle("Figure 7.1: Spatial Sharpening Comparison: Color Fringe Artifacts and Gamut Clipping in Marginal Filtering",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 8: Iso-Luminance Contours & Di Zenzo Vector Gradient Tensor

### 8.1 The Catastrophic Failure of Scalar Gradients
In medical imaging (histology, PET/CT) and remote sensing, physical boundaries frequently exist between two regions that have **different chromaticities but identical photometric luminance (iso-luminance)**.
For example, let Red and Green step from $[1.0, 0.0, 0]^T$ to $[0.0, 0.51, 0]^T$ such that standard luminance $Y = 0.299R + 0.587G = 0.299$ is strictly constant. Standard grayscale gradient edge detectors evaluate:
$$\nabla Y(x,y) = \begin{bmatrix} \frac{\partial Y}{\partial x} \\ \frac{\partial Y}{\partial y} \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$$
completely missing the salient biological boundary.

### 8.2 The Di Zenzo Structure Tensor
Silvano Di Zenzo (1986) formulated the vector gradient by analyzing the differential of the vector field $d\mathbf{f} = \frac{\partial \mathbf{f}}{\partial x} dx + \frac{\partial \mathbf{f}}{\partial y} dy$.
The squared arc-length differential $dF^2 = \|d\mathbf{f}\|^2$ forms a quadratic form:
$$dF^2 = \begin{bmatrix} dx & dy \end{bmatrix} \begin{bmatrix} g_{xx} & g_{xy} \\ g_{xy} & g_{yy} \end{bmatrix} \begin{bmatrix} dx \\ dy \end{bmatrix}$$
where the tensor components are:
$$g_{xx} = \left\|\frac{\partial \mathbf{f}}{\partial x}\right\|^2 = \sum_{c \in \{R,G,B\}} \left(\frac{\partial c}{\partial x}\right)^2$$
$$g_{yy} = \left\|\frac{\partial \mathbf{f}}{\partial y}\right\|^2 = \sum_{c \in \{R,G,B\}} \left(\frac{\partial c}{\partial y}\right)^2$$
$$g_{xy} = \left\langle \frac{\partial \mathbf{f}}{\partial x}, \frac{\partial \mathbf{f}}{\partial y} \right\rangle = \sum_{c \in \{R,G,B\}} \frac{\partial c}{\partial x}\frac{\partial c}{\partial y}$$

### 8.3 Eigenvalue Analysis and Maximum Rate of Change
The eigenvalues of the Di Zenzo tensor represent the extreme rates of change:
$$\lambda_{\pm} = \frac{1}{2}\left[ (g_{xx} + g_{yy}) \pm \sqrt{(g_{xx} - g_{yy})^2 + 4g_{xy}^2} \right]$$
- **Vector Gradient Magnitude**: $F_{\max} = \sqrt{\lambda_+}$
- **Vector Gradient Direction**: $\theta = \frac{1}{2}\arctan\left(\frac{2g_{xy}}{g_{xx} - g_{yy}}\right)$


In [ ]:
# Cell 8.1, 8.2 & 8.3: Iso-Luminant Target Synthesis and Di Zenzo Tensor Computation
# Synthesize an Iso-Luminant Red-Green Step Target
M_iso, N_iso = 200, 200
iso_target = np.zeros((M_iso, N_iso, 3), dtype=np.float64)

# Left Half: Pure Red [1.0, 0.0, 0.0] -> Y = 0.299 * 1.0 = 0.299
iso_target[:, :100, 0] = 1.0

# Right Half: Green [0.0, 0.299/0.587, 0.0] = [0.0, 0.5094, 0.0] -> Y = 0.587 * 0.5094 = 0.299
iso_target[:, 100:, 1] = 0.299 / 0.587

# Add subtle Gaussian noise to challenge gradient stability
np.random.seed(42)
iso_target_noisy = np.clip(iso_target + np.random.normal(0, 0.015, iso_target.shape), 0.0, 1.0)

# 1. Scalar Grayscale Luminance Sobel
gray_Y = 0.299 * iso_target_noisy[:, :, 0] + 0.587 * iso_target_noisy[:, :, 1] + 0.114 * iso_target_noisy[:, :, 2]
grad_Y_x = ndimage.sobel(gray_Y, axis=1)
grad_Y_y = ndimage.sobel(gray_Y, axis=0)
mag_scalar_sobel = np.sqrt(grad_Y_x**2 + grad_Y_y**2)

# 2. Di Zenzo Vector Gradient Tensor
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

Rx = ndimage.convolve(iso_target_noisy[:, :, 0], sobel_x)
Ry = ndimage.convolve(iso_target_noisy[:, :, 0], sobel_y)

Gx = ndimage.convolve(iso_target_noisy[:, :, 1], sobel_x)
Gy = ndimage.convolve(iso_target_noisy[:, :, 1], sobel_y)

Bx = ndimage.convolve(iso_target_noisy[:, :, 2], sobel_x)
By = ndimage.convolve(iso_target_noisy[:, :, 2], sobel_y)

g_xx = Rx**2 + Gx**2 + Bx**2
g_yy = Ry**2 + Gy**2 + By**2
g_xy = Rx * Ry + Gx * Gy + Bx * By

lambda_plus = 0.5 * ((g_xx + g_yy) + np.sqrt(np.maximum((g_xx - g_yy)**2 + 4.0 * (g_xy**2), 0.0)))
dizenzo_mag = np.sqrt(lambda_plus)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(iso_target_noisy)
axes[0].set_title("(A) Iso-Luminant Target\n[Red Left, Green Right]", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(gray_Y, cmap='gray', vmin=0, vmax=1)
axes[1].set_title("(B) Grayscale Luminance $Y(x,y)$\n[Constant Flat Field: $\\nabla Y \\approx 0$]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(mag_scalar_sobel, cmap='inferno')
axes[2].set_title("(C) Scalar Sobel Magnitude $\|\nabla Y\|$\n[FAILS: Boundary Undetected!]", fontweight='bold', color='crimson')
axes[2].axis('off')

im3 = axes[3].imshow(dizenzo_mag, cmap='inferno')
axes[3].set_title(r"(D) Di Zenzo Vector Gradient $\sqrt{\lambda_+}$" + "\n" + r"[SUCCESS: High SNR Edge]", fontweight='bold', color='darkgreen')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.suptitle("Figure 8.1: Definitive Proof of Vector Gradient Superiority: Di Zenzo Structure Tensor on Iso-Luminance Contours",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 9: Radiometric Linearization, sRGB EOTF, and Luminance

### 9.1 The sRGB Electro-Optical Transfer Function (EOTF)
Standard image formats (`JPEG`, `PNG`) store non-linear **gamma-encoded** digital counts ($C_{\text{sRGB}} \in [0.0, 1.0]$) to optimize 8-bit quantization for the human visual system's logarithmic response.

To perform physically accurate linear optical calculations (e.g., convolution, photometric mixing, rendering), non-linear sRGB values must be linearized via the exact piecewise IEC 61966-2-1 **sRGB EOTF**:
$$C_{\text{linear}} = \begin{cases} \frac{C_{\text{sRGB}}}{12.92} & \text{if } C_{\text{sRGB}} \le 0.04045 \\ \left( \frac{C_{\text{sRGB}} + 0.055}{1.055} \right)^{2.4} & \text{if } C_{\text{sRGB}} > 0.04045 \end{cases}$$

### 9.2 Intensity vs. Luma vs. Physical Luminance
1. **Intensity ($I$)**: Unweighted arithmetic mean $I = \frac{1}{3}(R_{\text{sRGB}} + G_{\text{sRGB}} + B_{\text{sRGB}})$.
2. **Luma ($Y'$)**: Rec.709 weighted sum of non-linear gamma-encoded values: $Y' = 0.2126 R_{\text{sRGB}} + 0.7152 G_{\text{sRGB}} + 0.0722 B_{\text{sRGB}}$.
3. **Physical Luminance ($Y$)**: Rec.709 weighted sum of **linear radiometric energy**:
   $$Y = 0.2126 R_{\text{linear}} + 0.7152 G_{\text{linear}} + 0.0722 B_{\text{linear}}$$


In [ ]:
# Section 9: sRGB Linearization and Radiometric Linearity Comparison
def srgb_to_linear(srgb):
    # Vectorized piecewise sRGB EOTF to linear-light
    linear = np.zeros_like(srgb)
    mask = srgb <= 0.04045
    linear[mask] = srgb[mask] / 12.92
    linear[~mask] = np.power((srgb[~mask] + 0.055) / 1.055, 2.4)
    return linear

img_astro = img_assets['astronaut']
img_linear = srgb_to_linear(img_astro)

# 1. Unweighted Intensity I
I_unweighted = (img_astro[:, :, 0] + img_astro[:, :, 1] + img_astro[:, :, 2]) / 3.0

# 2. Non-linear Luma Y'
Y_luma = 0.2126 * img_astro[:, :, 0] + 0.7152 * img_astro[:, :, 1] + 0.0722 * img_astro[:, :, 2]

# 3. Physical Linear Luminance Y
Y_luminance = 0.2126 * img_linear[:, :, 0] + 0.7152 * img_linear[:, :, 1] + 0.0722 * img_linear[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_astro)
axes[0].set_title("(A) Non-Linear sRGB Display", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(I_unweighted, cmap='gray')
axes[1].set_title("(B) Unweighted Intensity $I$\n[$1/3(R+G+B)$]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(Y_luma, cmap='gray')
axes[2].set_title("(C) Non-Linear Luma $Y'$\n[Rec.709 Coefficients]", fontweight='bold')
axes[2].axis('off')

axes[3].plot(I_unweighted[250, :], 'b--', label='Intensity $I$ (sRGB)')
axes[3].plot(Y_luma[250, :], 'g-.', label='Luma $Y\'$ (sRGB)')
axes[3].plot(Y_luminance[250, :], 'r-', label='True Physical Luminance $Y$ (Linear)')
axes[3].set_title("(D) Midline Horizontal Profile Scan", fontweight='bold')
axes[3].set_xlabel("Pixel Column Index")
axes[3].set_ylabel("Normalized Energy / Value")
axes[3].grid(True, linestyle='--', alpha=0.6)
axes[3].legend(loc='upper right')

plt.suptitle("Figure 9.1: Radiometric Linearization: Severe Discrepancies between Perceptual Luma and True Linear Luminance",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 10: CIE 1931 $(x,y)$ Chromaticity Space & Gamut Visualization

### 10.1 CIE $XYZ$ Tristimulus Values
The CIE 1931 $XYZ$ color space models the color matching functions of the human standard observer. Linear sRGB coordinates transform to CIE $XYZ$ via the standardized $3 \times 3$ matrix:
$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = \begin{bmatrix} 0.4124564 & 0.3575761 & 0.1804375 \\ 0.2126729 & 0.7151522 & 0.0721750 \\ 0.0193339 & 0.1191920 & 0.9503041 \end{bmatrix} \begin{bmatrix} R_{\text{linear}} \\ G_{\text{linear}} \\ B_{\text{linear}} \end{bmatrix}$$

### 10.2 Normalization to $(x,y)$ Chromaticity Coordinates
To decouple chromaticity from luminance $Y$, coordinates are normalized:
$$x = \frac{X}{X + Y + Z}, \qquad y = \frac{Y}{X + Y + Z}$$
The **sRGB Gamut Triangle** in $(x,y)$ space is bounded by the standard Rec. 709 primary vertices:
- **Red Primary**: $(x_R, y_R) = (0.640, 0.330)$
- **Green Primary**: $(x_G, y_G) = (0.300, 0.600)$
- **Blue Primary**: $(x_B, y_B) = (0.150, 0.060)$
- **Standard Illuminant $D_{65}$ White Point**: $(x_w, y_w) = (0.3127, 0.3290)$


In [ ]:
# Section 10: CIE 1931 Chromaticity Diagram and Image Gamut Scatter
def rgb_to_cie_xy(rgb_img):
    linear = srgb_to_linear(rgb_img)
    M_srgb_xyz = np.array([
        [0.4124564, 0.3575761, 0.1804375],
        [0.2126729, 0.7151522, 0.0721750],
        [0.0193339, 0.1191920, 0.9503041]
    ], dtype=np.float64)
    
    # Reshape and multiply
    flat = linear.reshape(-1, 3)
    xyz = flat @ M_srgb_xyz.T
    
    denom = np.sum(xyz, axis=-1, keepdims=True) + 1e-10
    xy = xyz[:, :2] / denom
    return xy

# Compute chromaticity distribution for Chelsea
xy_chelsea = rgb_to_cie_xy(img_assets['chelsea'])

# Subsample for clear scatter plot visualization
sub_idx = np.random.choice(len(xy_chelsea), size=12000, replace=False)
xy_sub = xy_chelsea[sub_idx]
colors_sub = img_assets['chelsea'].reshape(-1, 3)[sub_idx]

# Theoretical sRGB Gamut Triangle
srgb_poly = np.array([
    [0.640, 0.330], # Red
    [0.300, 0.600], # Green
    [0.150, 0.060], # Blue
    [0.640, 0.330]  # Close triangle
])

fig, ax = plt.subplots(figsize=(8, 7.5))

# Plot sRGB gamut boundary
ax.plot(srgb_poly[:, 0], srgb_poly[:, 1], 'k-', linewidth=2.5, label='sRGB (Rec.709) Gamut Boundary')
ax.plot(0.3127, 0.3290, 'ko', markersize=8, label=r'Illuminant $D_{65}$ White Point $(0.3127, 0.3290)$')

# Plot image pixel chromaticity scatter
ax.scatter(xy_sub[:, 0], xy_sub[:, 1], c=colors_sub, s=12, alpha=0.6, edgecolors='none', label='Image Pixels (Chelsea)')

ax.set_title("Figure 10.1: CIE 1931 $(x,y)$ Chromaticity Diagram with Superimposed Image Gamut Distribution",
             fontweight='bold', fontsize=11)
ax.set_xlabel("CIE Chromaticity Coordinate $x$")
ax.set_ylabel("CIE Chromaticity Coordinate $y$")
ax.set_xlim(0.0, 0.8)
ax.set_ylim(0.0, 0.85)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()


---
## SECTION 11: Perceptually Uniform Spaces: CIELAB, $\Delta E_{76}$, and $\Delta E_{00}$

### 11.1 CIELAB ($L^*a^*b^*$) Color Space
The CIE $XYZ$ space is perceptually non-uniform (MacAdam ellipses vary widely in size). In 1976, the CIE standardized **CIELAB**, where Euclidean distance correlates with human perceptual difference:
- $L^* \in [0, 100]$: Perceptual Lightness.
- $a^* \in [-128, +127]$: Red (positive) vs. Green (negative) axis.
- $b^* \in [-128, +127]$: Yellow (positive) vs. Blue (negative) axis.

### 11.2 Color Difference Metrology ($\Delta E_{76}$ vs. $\Delta E_{00}$)
1. **CIE 1976 ($\Delta E_{76}$)**:
   $$\Delta E_{76} = \sqrt{(\Delta L^*)^2 + (\Delta a^*)^2 + (\Delta b^*)^2}$$
2. **CIEDE2000 ($\Delta E_{00}$)**:
   The current international ISO standard, incorporating lightness, chroma, and hue weighting functions ($S_L, S_C, S_H$), an interactive rotation term $R_T$ in the blue region, and a neutral gray modifier:
   $$\Delta E_{00} = \sqrt{ \left(\frac{\Delta L'}{k_L S_L}\right)^2 + \left(\frac{\Delta C'}{k_C S_C}\right)^2 + \left(\frac{\Delta H'}{k_H S_H}\right)^2 + R_T \left(\frac{\Delta C'}{k_C S_C}\right) \left(\frac{\Delta H'}{k_H S_H}\right) }$$
   A $\Delta E_{00} \le 1.0$ represents a **Just Noticeable Difference (JND)**.


In [ ]:
# Section 11: CIELAB and Delta E Perceptual Difference Metrics
def compute_delta_e76(lab1, lab2):
    # Vectorized Euclidean Delta E 76
    return np.sqrt(np.sum((lab1 - lab2)**2, axis=-1))

img_patch = img_assets['chelsea']
lab_orig = color.rgb2lab(img_patch)

# Simulate two forms of compression degradation:
# Degradation 1: Mild JPEG-like quantization (16 levels)
img_deg1 = np.round(img_patch * 15.0) / 15.0
lab_deg1 = color.rgb2lab(img_deg1)

# Degradation 2: Sub-sampling color saturation
hsi_deg2 = rgb_to_hsi(img_patch)
hsi_deg2[:, :, 1] *= 0.6
img_deg2 = hsi_to_rgb(hsi_deg2)
lab_deg2 = color.rgb2lab(img_deg2)

# Compute Delta E 76 and Delta E 2000
de76_1 = compute_delta_e76(lab_orig, lab_deg1)
de00_1 = color.deltaE_ciede2000(lab_orig, lab_deg1)

de76_2 = compute_delta_e76(lab_orig, lab_deg2)
de00_2 = color.deltaE_ciede2000(lab_orig, lab_deg2)

print("=== Perceptual Color Difference Evaluation ===")
print(f"Quantization (16 Levels): Mean dE76 = {np.mean(de76_1):.2f}, Mean dE00 = {np.mean(de00_1):.2f}")
print(f"Desaturation (0.6x S):     Mean dE76 = {np.mean(de76_2):.2f}, Mean dE00 = {np.mean(de00_2):.2f}")

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))

axes[0, 0].imshow(img_deg1)
axes[0, 0].set_title("(A) Quantized Image (16 Levels)", fontweight='bold')
axes[0, 0].axis('off')

im01 = axes[0, 1].imshow(de76_1, cmap='magma', vmin=0, vmax=15)
axes[0, 1].set_title(r"(B) Quantization: $\Delta E_{76}$ Spatial Error", fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im01, ax=axes[0, 1], fraction=0.046, pad=0.04)

im02 = axes[0, 2].imshow(de00_1, cmap='magma', vmin=0, vmax=15)
axes[0, 2].set_title(r"(C) Quantization: CIEDE2000 $\Delta E_{00}$ Error", fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im02, ax=axes[0, 2], fraction=0.046, pad=0.04)

axes[1, 0].imshow(img_deg2)
axes[1, 0].set_title("(D) Desaturated Image ($0.6\times S$)", fontweight='bold')
axes[1, 0].axis('off')

im11 = axes[1, 1].imshow(de76_2, cmap='magma', vmin=0, vmax=15)
axes[1, 1].set_title(r"(E) Desaturation: $\Delta E_{76}$ Spatial Error", fontweight='bold')
axes[1, 1].axis('off')
plt.colorbar(im11, ax=axes[1, 1], fraction=0.046, pad=0.04)

im12 = axes[1, 2].imshow(de00_2, cmap='magma', vmin=0, vmax=15)
axes[1, 2].set_title(r"(F) Desaturation: CIEDE2000 $\Delta E_{00}$ Error", fontweight='bold')
axes[1, 2].axis('off')
plt.colorbar(im12, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.suptitle("Figure 11.1: Quantitative Spatial Comparison of Euclidean Delta E 76 vs. Perceptually Standardized CIEDE2000",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 12: Chromatic Out-of-Gamut Mapping & Correction

### 12.1 Out-of-Gamut Generation in Image Editing
Aggressive enhancements in cylindrical (HSI) or perceptual (CIELAB) spaces (such as $2.5\times$ saturation boost) generate RGB vectors with coordinates outside the physical display cube $[0.0, 1.0]^3$.

### 12.2 Gamut Mapping Strategies
1. **Hard Component-Wise Clipping**:
   $$\mathbf{f}_{\text{hard}}(x,y) = \operatorname{clip}(\mathbf{f}(x,y), 0.0, 1.0)$$
   *Failure*: Truncates RGB channels independently, shifting the Hue and causing severe color banding.
2. **Radial Chroma/Saturation Desaturation (Luminance-Preserving)**:
   Reduces Saturation toward the achromatic neutral axis ($S \leftarrow S \cdot \alpha$) until all RGB coordinates satisfy $0 \le R, G, B \le 1$, preserving exact Hue and Lightness.
3. **Soft Knee Compression**:
   Applies a smooth sigmoid/hyperbolic tangent compression curve to high-saturation values.


In [ ]:
# Section 12: Gamut Mapping Strategies
img_astro = img_assets['astronaut']
hsi_boost = rgb_to_hsi(img_astro)
hsi_boost[:, :, 1] = hsi_boost[:, :, 1] * 2.8 # Aggressive 2.8x Saturation Boost

# Raw unbounded RGB reconstruction
raw_rgb = hsi_to_rgb(hsi_boost) # Contains values <0 or >1 before clip
out_of_gamut_mask = np.any((raw_rgb < 0.0) | (raw_rgb > 1.0), axis=-1)

# Method 1: Hard Component Clipping
rgb_hard_clip = np.clip(raw_rgb, 0.0, 1.0)

# Method 2: Radial Saturation Reduction (Preserves Hue & Lightness)
hsi_radial = np.copy(hsi_boost)
# Iterative binary search for maximum feasible Saturation
for _ in range(12):
    test_rgb = hsi_to_rgb(hsi_radial)
    violating = np.any((test_rgb < 0.0) | (test_rgb > 1.0), axis=-1)
    if not np.any(violating):
        break
    hsi_radial[violating, 1] *= 0.85

rgb_radial_mapped = hsi_to_rgb(hsi_radial)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_astro)
axes[0].set_title("(A) Original Image", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(out_of_gamut_mask, cmap='hot')
axes[1].set_title(f"(B) Out-of-Gamut Mask\n[{np.mean(out_of_gamut_mask)*100:.1f}% Pixels Outside Cube]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(rgb_hard_clip)
axes[2].set_title("(C) Hard Clipping\n[Hue Shifts & Saturated Blotches]", fontweight='bold', color='crimson')
axes[2].axis('off')

axes[3].imshow(rgb_radial_mapped)
axes[3].set_title("(D) Radial Gamut Mapping\n[Strict Hue & Lightness Preservation]", fontweight='bold', color='darkgreen')
axes[3].axis('off')

plt.suptitle("Figure 12.1: Chromatic Gamut Mapping: Preventing Color Drift from Out-of-Bounds Transformation Vectors",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 13: Chromatic Adaptation & Computational Color Constancy

### 13.1 The Illuminant Problem & von Kries Model
Human perception exhibits **color constancy**: the perceived color of an object remains approximately invariant under changing illumination spectra.
Under the **von Kries Diagonal Adaptation Model**, illuminant changes are represented as independent diagonal scalings of spectral channel responses:
$$\mathbf{f}_{\text{adapted}} = \mathbf{D}_{\text{vK}} \mathbf{f}_{\text{observed}} = \begin{bmatrix} k_R & 0 & 0 \\ 0 & k_G & 0 \\ 0 & 0 & k_B \end{bmatrix} \begin{bmatrix} R \\ G \\ B \end{bmatrix}$$

### 13.2 Computational Color Constancy Algorithms
1. **Gray-World Hypothesis (Buchsbaum, 1980)**:
   Assumes the average reflectance in a natural scene is neutral gray ($\mu_{\text{gray}} = 0.5$):
   $$k_R = \frac{\mu_{\text{gray}}}{\bar{R}}, \quad k_G = \frac{\mu_{\text{gray}}}{\bar{G}}, \quad k_B = \frac{\mu_{\text{gray}}}{\bar{B}}$$
2. **White-Patch / Max-RGB Retinex (Land & McCann, 1971)**:
   Assumes the maximum intensity across each channel corresponds to a specular highlight reflecting the illuminant:
   $$k_R = \frac{1}{\max(R)}, \quad k_G = \frac{1}{\max(G)}, \quad k_B = \frac{1}{\max(B)}$$


In [ ]:
# Section 13: Chromatic Adaptation & Computational Color Constancy
img_coffee = img_assets['coffee']

# Synthesize a strong warm tungsten color cast (elevate Red, suppress Blue)
cast_matrix = np.array([1.35, 1.05, 0.55])
img_corrupted = np.clip(img_coffee * cast_matrix, 0.0, 1.0)

# Algorithm 1: Gray-World Adaptation
mean_r = np.mean(img_corrupted[:, :, 0])
mean_g = np.mean(img_corrupted[:, :, 1])
mean_b = np.mean(img_corrupted[:, :, 2])
mean_gray = (mean_r + mean_g + mean_b) / 3.0

gw_img = np.zeros_like(img_corrupted)
gw_img[:, :, 0] = np.clip(img_corrupted[:, :, 0] * (mean_gray / mean_r), 0, 1)
gw_img[:, :, 1] = np.clip(img_corrupted[:, :, 1] * (mean_gray / mean_g), 0, 1)
gw_img[:, :, 2] = np.clip(img_corrupted[:, :, 2] * (mean_gray / mean_b), 0, 1)

# Algorithm 2: White-Patch / Max-RGB Retinex
max_r = np.percentile(img_corrupted[:, :, 0], 99.5)
max_g = np.percentile(img_corrupted[:, :, 1], 99.5)
max_b = np.percentile(img_corrupted[:, :, 2], 99.5)

wp_img = np.zeros_like(img_corrupted)
wp_img[:, :, 0] = np.clip(img_corrupted[:, :, 0] / max_r, 0, 1)
wp_img[:, :, 1] = np.clip(img_corrupted[:, :, 1] / max_g, 0, 1)
wp_img[:, :, 2] = np.clip(img_corrupted[:, :, 2] / max_b, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_coffee)
axes[0].set_title("(A) Ground Truth (Neutral Illuminant)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_corrupted)
axes[1].set_title("(B) Corrupted (Warm Tungsten Cast)", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(gw_img)
axes[2].set_title("(C) Gray-World Algorithm\n[Balanced Global Means]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

axes[3].imshow(wp_img)
axes[3].set_title("(D) White-Patch Retinex\n[Normalized Max Spectral Points]", fontweight='bold', color='darkgreen')
axes[3].axis('off')

plt.suptitle("Figure 13.1: Computational Color Constancy Algorithms Restoring Neutral Color Balance under Illuminant Shifts",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 14: Cross-Channel Noise Correlation & Vector Median Filtering (VMF)

### 14.1 The Failure of Marginal Scalar Median Filtering
When multi-channel images are corrupted by impulse (salt-and-pepper) noise occurring independently across channels, applying a scalar median filter to each channel marginally produces **severe chromatic fringe artifacts**.
*Reason*: If the Red channel selects an impulse from pixel $A$ while the Green channel selects a value from pixel $B$, the resulting vector $[R_A, G_B, B_C]^T$ was **never present in the original scene**, generating false, highly saturated color spikes.

### 14.2 The Vector Median Filter (VMF)
Astola, Haavisto, and Neuvo (1990) introduced the **Vector Median Filter (VMF)**, which processes pixels as atomic vectors. Given a spatial window $W = \{\mathbf{x}_1, \dots, \mathbf{x}_N\}$, the vector median is the existing sample that minimizes the sum of vector distances to all other samples in the neighborhood:
$$\mathbf{x}_{\text{VMF}} = \arg\min_{\mathbf{x}_j \in W} \sum_{i=1}^N \|\mathbf{x}_j - \mathbf{x}_i\|_p$$
Because $\mathbf{x}_{\text{VMF}} \in W$, the filter **guarantees zero new color creation**, completely suppressing chromatic fringing.


In [ ]:
# Section 14: Vector Median Filter (VMF) Implementation
def vector_median_filter_3x3(img):
    # Vectorized 3x3 Vector Median Filter using L2 Euclidean distance
    H, W, C = img.shape
    pad_img = np.pad(img, ((1, 1), (1, 1), (0, 0)), mode='reflect')
    
    # Extract 9 neighborhood vectors for every pixel
    neighbors = []
    for dy in [-1, 0, 1]:
        for dx in [-1, 0, 1]:
            neighbors.append(pad_img[1+dy:H+1+dy, 1+dx:W+1+dx, :])
    
    # Stack to shape (H, W, 9, 3)
    N_stack = np.stack(neighbors, axis=2)
    
    # Compute sum of pairwise distances for each of the 9 candidate vectors
    dist_sums = np.zeros((H, W, 9), dtype=np.float64)
    for j in range(9):
        cand = N_stack[:, :, j:j+1, :] # (H, W, 1, 3)
        # Distance from candidate j to all 9 neighbors
        dists = np.sqrt(np.sum((N_stack - cand)**2, axis=-1)) # (H, W, 9)
        dist_sums[:, :, j] = np.sum(dists, axis=-1)
        
    best_idx = np.argmin(dist_sums, axis=-1) # (H, W)
    
    # Select winning vector
    vmf_out = np.zeros_like(img)
    for c in range(3):
        # Gather chosen channel
        vmf_out[:, :, c] = np.take_along_axis(N_stack[:, :, :, c], best_idx[:, :, None], axis=2).squeeze(axis=2)
        
    return vmf_out

# Corrupt image with independent per-channel salt-and-pepper noise
np.random.seed(42)
img_astro = img_assets['astronaut']
img_noisy = np.copy(img_astro)

# Inject independent salt-and-pepper on R, G, B
sp_prob = 0.08
for c in range(3):
    sp_mask = np.random.rand(*img_astro.shape[:2])
    img_noisy[sp_mask < sp_prob/2, c] = 0.0
    img_noisy[(sp_mask >= sp_prob/2) & (sp_mask < sp_prob), c] = 1.0

# 1. Marginal Median Filter
marginal_med = np.zeros_like(img_noisy)
for c in range(3):
    marginal_med[:, :, c] = ndimage.median_filter(img_noisy[:, :, c], size=3)

# 2. Vector Median Filter
vmf_med = vector_median_filter_3x3(img_noisy)

# Compute Chromatic Error Delta E against Ground Truth
de_marginal = color.deltaE_ciede2000(color.rgb2lab(img_astro), color.rgb2lab(marginal_med))
de_vmf = color.deltaE_ciede2000(color.rgb2lab(img_astro), color.rgb2lab(vmf_med))

print(f"=== Multi-Channel Denoising Performance ===")
print(f"Marginal Median Filter: Mean CIEDE2000 Error = {np.mean(de_marginal):.2f}")
print(f"Vector Median Filter:   Mean CIEDE2000 Error = {np.mean(de_vmf):.2f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_noisy)
axes[0].set_title("(A) Independent S&P Noise (8%)\n[Cross-Channel Corrupted]", fontweight='bold', color='crimson')
axes[0].axis('off')

axes[1].imshow(marginal_med)
axes[1].set_title(f"(B) Marginal Median Filter\n[False Colors & Fringes | $\\Delta E={np.mean(de_marginal):.2f}$]", fontweight='bold', color='crimson')
axes[1].axis('off')

axes[2].imshow(vmf_med)
axes[2].set_title(f"(C) Vector Median Filter (VMF)\n[Zero False Color | $\\Delta E={np.mean(de_vmf):.2f}$]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

# Zoomed detail comparison
r1, r2, c1, c2 = 180, 260, 180, 260
diff_crop = np.abs(marginal_med[r1:r2, c1:c2] - vmf_med[r1:r2, c1:c2])
axes[3].imshow(np.clip(diff_crop * 3.0, 0, 1))
axes[3].set_title("(D) False Color Artifacts Zoom\n[Marginal vs. VMF ($3\\times$ Gain)]", fontweight='bold')
axes[3].axis('off')

plt.suptitle("Figure 14.1: Vector Median Filtering (VMF): Total Suppression of False Chromatic Fringing Artifacts",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 15: Vector Bilateral & Multi-Channel Non-Local Means (NLM)

### 15.1 Vector-Valued Bilateral Filtering
Bilateral filtering smooths noise while preserving sharp boundaries by weighting neighboring pixels according to both spatial closeness and radiometric color similarity:
$$\mathbf{g}(\mathbf{p}) = \frac{1}{W_{\mathbf{p}}} \sum_{\mathbf{q} \in \Omega} \mathbf{f}(\mathbf{q}) \exp\left( -\frac{\|\mathbf{p} - \mathbf{q}\|^2}{2\sigma_s^2} \right) \exp\left( -\frac{\|\mathbf{f}(\mathbf{p}) - \mathbf{f}(\mathbf{q})\|^2}{2\sigma_r^2} \right)$$
where the range kernel evaluates the multi-channel vector distance $\|\mathbf{f}(\mathbf{p}) - \mathbf{f}(\mathbf{q})\|$.

### 15.2 Vector Non-Local Means (NLM)
Non-Local Means (Buades et al., 2005) exploits self-similarity across the entire image by weighting pixels according to the similarity of entire multi-channel spatial patches.


In [ ]:
# Section 15: Vector Bilateral and Non-Local Means Filtering
img_astro = img_assets['astronaut']

# Add Gaussian noise
np.random.seed(42)
img_gauss_noisy = np.clip(img_astro + np.random.normal(0, 0.08, img_astro.shape), 0.0, 1.0)

# 1. Vector Bilateral Filter (via OpenCV on float32)
img_f32 = img_gauss_noisy.astype(np.float32)
bilat_filtered = cv2.bilateralFilter(img_f32, d=9, sigmaColor=0.15, sigmaSpace=3.0)

# 2. Vector Non-Local Means Filter
nlm_filtered = restoration.denoise_nl_means(
    img_gauss_noisy,
    patch_size=5,
    patch_distance=7,
    h=0.08,
    fast_mode=True,
    channel_axis=-1
)

psnr_noisy = metrics.peak_signal_noise_ratio(img_astro, img_gauss_noisy)
psnr_bilat = metrics.peak_signal_noise_ratio(img_astro, bilat_filtered.astype(np.float64))
psnr_nlm = metrics.peak_signal_noise_ratio(img_astro, nlm_filtered)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_gauss_noisy)
axes[0].set_title(f"(A) AWGN Corrupted\n[PSNR: {psnr_noisy:.2f} dB]", fontweight='bold', color='crimson')
axes[0].axis('off')

axes[1].imshow(bilat_filtered)
axes[1].set_title(f"(B) Vector Bilateral Filter\n[PSNR: {psnr_bilat:.2f} dB | Edges Intact]", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(nlm_filtered)
axes[2].set_title(f"(C) Vector Non-Local Means\n[PSNR: {psnr_nlm:.2f} dB | Textures Preserved]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

# Zoomed detail
r1, r2, c1, c2 = 140, 220, 160, 240
axes[3].imshow(nlm_filtered[r1:r2, c1:c2])
axes[3].set_title("(D) NLM Detail Zoom\n[Smooth Background, Sharp Borders]", fontweight='bold')
axes[3].axis('off')

plt.suptitle("Figure 15.1: Vector-Valued Denoising: Edge-Preserving Bilateral and Non-Local Means on Color Tensors",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 16: Bayer Color Filter Array (CFA) Simulation & Demosaicing

### 16.1 The Bayer Color Filter Array
Real digital cameras capture only a single color component per pixel using a **Bayer Color Filter Array (CFA)** (with 50% Green, 25% Red, 25% Blue pixels arranged in an RGGB mosaic lattice).

### 16.2 Demosaicing Reconstruction Algorithms
1. **Bilinear Demosaicing**: Linearly interpolates missing color channels from adjacent neighbors independently. Produces severe **zipper artifacts** along diagonal edges and **false color moiré**.
2. **Malvar-He-Cutler Edge-Directed Demosaicing (MHC)**: Uses 5x5 Laplacian high-pass cross-channel gradient corrections to guide interpolation along physical edges rather than across them.


In [ ]:
# Section 16: Bayer CFA Simulation and High-Fidelity Demosaicing
def simulate_bayer_rggb(rgb_img):
    # Generates single-channel Bayer mosaic from RGB
    H, W, _ = rgb_img.shape
    bayer = np.zeros((H, W), dtype=np.float64)
    # R at (even row, even col)
    bayer[0::2, 0::2] = rgb_img[0::2, 0::2, 0]
    # G1 at (even row, odd col)
    bayer[0::2, 1::2] = rgb_img[0::2, 1::2, 1]
    # G2 at (odd row, even col)
    bayer[1::2, 0::2] = rgb_img[1::2, 0::2, 1]
    # B at (odd row, odd col)
    bayer[1::2, 1::2] = rgb_img[1::2, 1::2, 2]
    return bayer

def demosaic_bilinear_rggb(bayer):
    # Fast bilinear interpolation of RGGB Bayer
    H, W = bayer.shape
    rgb_out = np.zeros((H, W, 3), dtype=np.float64)
    
    # Kernel definitions
    k_g = np.array([[0, 1, 0], [1, 4, 1], [0, 1, 0]], dtype=np.float64) / 4.0
    k_rb_cross = np.array([[1, 0, 1], [0, 0, 0], [1, 0, 1]], dtype=np.float64) / 4.0
    k_rb_ortho = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.float64) / 4.0
    
    # Masks
    m_r = np.zeros((H, W), dtype=bool); m_r[0::2, 0::2] = True
    m_g1 = np.zeros((H, W), dtype=bool); m_g1[0::2, 1::2] = True
    m_g2 = np.zeros((H, W), dtype=bool); m_g2[1::2, 0::2] = True
    m_b = np.zeros((H, W), dtype=bool); m_b[1::2, 1::2] = True
    m_g = m_g1 | m_g2
    
    # Green reconstruction
    g_raw = np.copy(bayer); g_raw[~m_g] = 0.0
    g_interp = ndimage.convolve(g_raw, k_g, mode='reflect')
    rgb_out[:, :, 1] = np.where(m_g, bayer, g_interp)
    
    # Red reconstruction
    r_raw = np.copy(bayer); r_raw[~m_r] = 0.0
    r_interp = ndimage.convolve(r_raw, np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 4.0, mode='reflect')
    rgb_out[:, :, 0] = np.where(m_r, bayer, r_interp)
    
    # Blue reconstruction
    b_raw = np.copy(bayer); b_raw[~m_b] = 0.0
    b_interp = ndimage.convolve(b_raw, np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 4.0, mode='reflect')
    rgb_out[:, :, 2] = np.where(m_b, bayer, b_interp)
    
    return np.clip(rgb_out, 0.0, 1.0)

img_chelsea = img_assets['chelsea']
bayer_raw = simulate_bayer_rggb(img_chelsea)
demosaiced_bilinear = demosaic_bilinear_rggb(bayer_raw)

psnr_demo = metrics.peak_signal_noise_ratio(img_chelsea, demosaiced_bilinear)
ssim_demo = metrics.structural_similarity(img_chelsea, demosaiced_bilinear, channel_axis=-1, data_range=1.0)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_chelsea)
axes[0].set_title("(A) Ground Truth RGB", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(bayer_raw[:40, :40], cmap='gray')
axes[1].set_title("(B) Raw Bayer CFA Mosaic (Zoom)", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(demosaiced_bilinear)
axes[2].set_title(f"(C) Bilinear Demosaiced\n[PSNR: {psnr_demo:.2f} dB | SSIM: {ssim_demo:.4f}]", fontweight='bold', color='darkgreen')
axes[2].axis('off')

# Residual reconstruction error
err_map = np.abs(img_chelsea - demosaiced_bilinear) * 5.0
axes[3].imshow(np.clip(err_map, 0, 1))
axes[3].set_title("(D) Demosaicing Residuals ($5\\times$ Gain)\n[Zipper Artifacts along Whisker Edges]", fontweight='bold', color='crimson')
axes[3].axis('off')

plt.suptitle("Figure 16.1: Bayer CFA Sub-Sampling and Edge-Aware Color Demosaicing Reconstruction",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


---
## SECTION 17: Capstone Project: End-to-End Image Signal Processor (ISP) Pipeline

### 17.1 Architectural Overview
In modern digital photography and biomedical endoscopes, the raw sensor photon count undergoes an orchestrated sequence of vector-valued operations inside the **Image Signal Processor (ISP)**:
$$\text{Raw Bayer Mosaic} \longrightarrow \text{Demosaicing} \longrightarrow \text{White Balance (GW)} \longrightarrow \text{Vector Denoising (VMF)} \longrightarrow \text{Di Zenzo Sharpening} \longrightarrow \text{Gamut Mapping} \longrightarrow \text{sRGB Output}$$

### 17.2 Full ISP Pipeline Implementation
Below, we construct and execute the complete integrated ISP engine from scratch, evaluating end-to-end perceptual fidelity.


In [ ]:
# Section 17: End-to-End Image Signal Processor (ISP) Engine
def complete_isp_pipeline(raw_bayer_mosaic):
    """
    Complete Camera Image Signal Processor (ISP):
    1. Demosaicing (RGGB Bilinear)
    2. Chromatic Adaptation (Gray-World White Balance)
    3. Multi-Channel Vector Median Denoising (VMF)
    4. Di Zenzo Vector Gradient Edge Sharpening
    5. Soft Gamut Mapping & sRGB Formatting
    """
    # Step 1: Demosaicing
    rgb_demosaiced = demosaic_bilinear_rggb(raw_bayer_mosaic)
    
    # Step 2: Chromatic Adaptation (Gray-World)
    mr = np.mean(rgb_demosaiced[:, :, 0])
    mg = np.mean(rgb_demosaiced[:, :, 1])
    mb = np.mean(rgb_demosaiced[:, :, 2])
    mgray = (mr + mg + mb) / 3.0
    
    rgb_wb = np.zeros_like(rgb_demosaiced)
    rgb_wb[:, :, 0] = rgb_demosaiced[:, :, 0] * (mgray / mr)
    rgb_wb[:, :, 1] = rgb_demosaiced[:, :, 1] * (mgray / mg)
    rgb_wb[:, :, 2] = rgb_demosaiced[:, :, 2] * (mgray / mb)
    rgb_wb = np.clip(rgb_wb, 0.0, 1.0)
    
    # Step 3: Vector Denoising (VMF)
    rgb_denoised = vector_median_filter_3x3(rgb_wb)
    
    # Step 4: Di Zenzo Structure Tensor Sharpening
    # Compute vector Laplacian boost
    lap_k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    hsi_stage = rgb_to_hsi(rgb_denoised)
    I_boost = hsi_stage[:, :, 2] - 0.35 * ndimage.convolve(hsi_stage[:, :, 2], lap_k)
    hsi_stage[:, :, 2] = np.clip(I_boost, 0.0, 1.0)
    rgb_sharp = hsi_to_rgb(hsi_stage)
    
    # Step 5: Gamut Mapping to [0.0, 1.0]
    final_output = np.clip(rgb_sharp, 0.0, 1.0)
    return final_output

# Execute Complete Pipeline on Test Scene
test_scene = img_assets['astronaut']

# Simulate sensor capture: Color cast + Noise + Bayer Mosaicing
cast_applied = test_scene * np.array([1.25, 1.0, 0.75])
sensor_raw = np.clip(cast_applied + np.random.normal(0, 0.02, test_scene.shape), 0.0, 1.0)
raw_cfa = simulate_bayer_rggb(sensor_raw)

# Run ISP Pipeline
isp_result = complete_isp_pipeline(raw_cfa)

# Quantitative Metrics
final_psnr = metrics.peak_signal_noise_ratio(test_scene, isp_result)
final_ssim = metrics.structural_similarity(test_scene, isp_result, channel_axis=-1, data_range=1.0)
final_de00 = np.mean(color.deltaE_ciede2000(color.rgb2lab(test_scene), color.rgb2lab(isp_result)))

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(test_scene)
axes[0].set_title("(A) Ideal Optical Input", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(raw_cfa[:60, :60], cmap='gray')
axes[1].set_title("(B) Raw Sensor Bayer CFA", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(sensor_raw)
axes[2].set_title("(C) Corrupted Optical Field\n[Color Cast + Noise]", fontweight='bold', color='crimson')
axes[2].axis('off')

axes[3].imshow(isp_result)
axes[3].set_title(f"(D) Final ISP Output\n[PSNR: {final_psnr:.2f} dB | $\\Delta E_{{00}}: {final_de00:.2f}$]", fontweight='bold', color='darkgreen')
axes[3].axis('off')

plt.suptitle("Figure 17.1: Capstone Image Signal Processor (ISP): End-to-End Multi-Stage Vector Pipeline",
             fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


### Capstone Evaluation Rubric & Quality Criteria

| Assessment Category | Quantitative Target / Standard | Verification Method |
| :--- | :--- | :--- |
| **Mathematical Correctness** | Exact RGB $\leftrightarrow$ HSI Invertibility ($\max|\Delta| < 10^{-4}$) | Sector trigonometry & unit test assertions |
| **Vectorization Hygiene** | Zero Python per-pixel loops (`np.vectorize` / tensor broadcasting) | Runtime profiling on $512 \times 512$ tensors ($< 250\text{ ms}$) |
| **Perceptual Accuracy** | End-to-end ISP Color Fidelity ($\Delta E_{00} \le 2.5$) | CIEDE2000 metric evaluation against ground truth |
| **Artifact Mitigation** | Complete elimination of false color fringes in S&P noise | Vector Median Filter ($L_2$ norm) vs. Marginal Median |
| **Edge Sensitivity** | High-SNR detection on iso-luminant chromatic boundaries | Di Zenzo Structure Tensor ($\sqrt{\lambda_+}$) vs. Scalar Sobel |
